# 手撕 SentencePiece 综述与简版实现

## 背景
SentencePiece 是统一的分词框架，支持 BPE/Unigram/Word/Char 四种模型。
关键特性：将空格转为特殊字符 ▁（U+2581），实现语言无关的分词。

## 考察点
- ▁ (meta space) 的作用
- BPE vs Unigram 的选择
- encode/decode 的可逆性

In [ ]:
import math
from collections import Counter

class SimpleSentencePiece:
    def __init__(self, model_type: str = "unigram", vocab_size: int = 100) -> None:
        self.model_type = model_type
        self.vocab_size = vocab_size
        self.vocab = {}  # {token: id}
        self.id2token = {}
        self.scores = {}  # {token: log_prob} for unigram
        self.merges = []  # for bpe
        self.unk = "<unk>"
        self.sp_marker = "▁"  # ▁

    def _preprocess(self, text: str) -> str:
        # 空格 → ▁，添加前缀
        return self.sp_marker + self.sp_marker.join(text.split())

    def _postprocess(self, tokens: str) -> str:
        # ▁ → 空格
        return "".join(tokens).replace(self.sp_marker, " ").strip()

    def train(self, texts: list) -> None:
        processed = [self._preprocess(t) for t in texts]
        if self.model_type == "unigram":
            substr_freqs = Counter()
            for text in processed:
                for i in range(len(text)):
                    for j in range(i + 1, min(len(text) + 1, i + 10)):
                        substr_freqs[text[i:j]] += 1
            top = substr_freqs.most_common(self.vocab_size)
            total = sum(f for _, f in top)
            for i, (tok, freq) in enumerate(top):
                self.vocab[tok] = i
                self.scores[tok] = math.log(freq / total)
        self.id2token = {v: k for k, v in self.vocab.items()}
        if self.unk not in self.vocab:
            self.vocab[self.unk] = len(self.vocab)
            self.id2token[self.vocab[self.unk]] = self.unk

    def encode(self, text: str) -> torch.Tensor:
        processed = self._preprocess(text)
        if self.model_type == "unigram":
            # Viterbi
            n = len(processed)
            dp = [-float('inf')] * (n + 1)
            dp[0] = 0.0
            back = [0] * (n + 1)
            for end in range(1, n + 1):
                for start in range(max(0, end - 10), end):
                    sub = processed[start:end]
                    if sub in self.scores and dp[start] + self.scores[sub] > dp[end]:
                        dp[end] = dp[start] + self.scores[sub]
                        back[end] = start
            tokens = []
            pos = n
            while pos > 0:
                tokens.append(processed[back[pos]:pos])
                pos = back[pos]
            return tokens[::-1]
        return list(processed)

    def decode(self, tokens: str) -> torch.Tensor:
        return self._postprocess(tokens)

In [ ]:
# 验证 SentencePiece
texts = ["hello world", "hello hello", "world peace"]
sp = SimpleSentencePiece(model_type="unigram", vocab_size=80)
sp.train(texts)
encoded = sp.encode("hello world")
decoded = sp.decode(encoded)
print(f"encode('hello world'): {encoded}")
print(f"decode: '{decoded}'")
assert len(encoded) > 0, "编码非空"
assert "hello" in decoded and "world" in decoded, "解码应包含原词"
# 验证 ▁ 标记
assert any("▁" in t for t in encoded), "应包含 ▁ 空格标记"
print("✅ SentencePiece encode/decode 验证通过")